# Fish Toxicty Further Algorithms & Scaling
In this dataset, each row is an experiment on the rainbow trout. The columns show what chemicals were present, how long the fish were left in the toxic water, the temperature of the water, etc. With that, our target variable is "result_conc1_mean_mol_log" which is the The base-10 logarithm of the mean molar concentration log10(mol/L) required to reach the specified experimental toxicity endpoint (LC50)

In [1]:
import pandas as pd

# read csv
df = pd.read_csv(
    "../adore_dataset/processed/s-F2F-1_mortality.csv",
    dtype=str
)
split_def = df["split_occurrence"]

# every column that starts with tax is about the fish, but this dataset is only about one fish
starts_with_tax = [col for col in df.columns if col.startswith("tax")]

# every column that starts with split is about different ways the data was split
starts_with_split = [col for col in df.columns if col.startswith("split")]

cols_to_drop = [
    # LC50 results (answers are computed from these values)
    "result_conc1_mean", "result_conc1_mean_mol", "result_conc1_mean_log",
    "result_conc1_mean_mol_log", "result_conc1_mean_op",
    
    # IDs and names
    "test_id", "result_id", "reference_number", "test_cas", "test_cas_name", "chem_name",
    "chem_dtxsid", "chem_dtxcid", "chem_inchi", "chem_inchikey", "chem_pcp_cid",
    "chem_pcp_inchikey", "chem_pcp_inchi", "chem_pcp_iupac_name",
    "chem_pcp_can_smiles", "chem_rdkit_can_smiles", "chem_sf",

    # finger prints
    "chem_pcp_fp", "chem_MACCS_fp", "chem_Morgan_fp", "chem_ToxPrint_fp",

    # not useful
    "test_location", "test_control_type", "test_application_freq_unit",
    "test_organism_lifestage", "result_effect", "result_endpoint", "result_conc1_type",
    "test_media_type", "chem_ws_binary", "chem_mol2vec_allowed",
    "media_ph_mean", "media_temperature_mean", "chem_pka_median", "species_number"
    ]

# add columns that start with tax or split to the columns to drop
cols_to_drop.extend(starts_with_tax)
cols_to_drop.extend(starts_with_split)

print(df.shape)
df = df.drop(columns=cols_to_drop)
df["split_occurrence"] = split_def
df.shape


(4641, 1106)


(4641, 1033)

In [2]:
# define numeric columns
numeric_cols = []

for col in df.columns:
    converted = pd.to_numeric(df[col], errors="coerce")

    if converted.notna().sum() == df[col].notna().sum():
        numeric_cols.append(col)



In [3]:
# define categorical columns
categorical_cols = [col for col in df.columns if col not in numeric_cols]
categorical_cols


['test_exposure_type', 'split_occurrence']

In [4]:
# define test and train sets
test_set = df[df["split_occurrence"] == "test"]
train_set = df[df["split_occurrence"] != "test"]


In [19]:
from sklearn.model_selection import PredefinedSplit
import numpy as np

# split folds according to the pre-defined split in "split occurence"
test_fold = train_set["split_occurrence"].astype(int).values
ps=PredefinedSplit(test_fold)

# define X, y for train and test sets
X_train = train_set.drop(
    columns=["split_occurrence", "result_conc1_mean_binary","test_exposure_type"]
)

y_train = train_set["result_conc1_mean_binary"]

X_test = test_set.drop(
    columns=["split_occurrence", "result_conc1_mean_binary","test_exposure_type"]
)

y_test = test_set["result_conc1_mean_binary"]

X_train.shape


(3928, 1030)

In [20]:
# Check for missing values
has_missing = X_train.isna().sum()
has_missing[has_missing > 0]


#X_train = X_train.replace("NR", np.nan)
#has_missing[has_missing > 0] # has missing values in "test_exposure_type" (i dont really want to include this)



Series([], dtype: int64)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

 
classifiers = {
    "Nearest Neighbors": KNeighborsClassifier(n_neighbors = 3),
    "Linear SVM": SVC(kernel="linear", C=0.025, random_state=42),
    "RBF SVM": SVC(gamma=2, C=1, random_state=42),
    #"Gaussian Process": GaussianProcessClassifier(1.0 * RBF(1.0), random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(max_depth=5, n_estimators=10, max_features=1, random_state=42),
    "Neural Net": MLPClassifier(alpha=1, max_iter=1000, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
    #"QDA": QuadraticDiscriminantAnalysis(),
}


for name, model in classifiers.items():

    pipeline = make_pipeline(
        StandardScaler(),
        model
    )

    scores = cross_val_score(pipeline, X_train, y_train, cv=ps)
    print('--------------------')
    print(name)
    print(f"score:  {scores.mean():.3f}  std: {scores.std():.3f}")


--------------------
Nearest Neighbors
score:  0.765  std: 0.073
--------------------
Linear SVM
score:  0.766  std: 0.059
--------------------
RBF SVM
score:  0.595  std: 0.066
--------------------
Decision Tree
score:  0.674  std: 0.064
--------------------
Random Forest
score:  0.766  std: 0.061
